## Part 3.1: Create, Load, and Validate Data Quality Rules

### Objective

Create or load the persistent data-quality rules used to evaluate the contents of the Cyclistic source files.

Part 2 verified the structural schema of the source data. Part 3 evaluates whether the values inside that structure are complete, valid, consistent, and suitable for further processing.

This section defines the quality contract only. It does not scan source files or calculate quality metrics.

### Approved Schema Dependency

Data-quality rules must be associated with an approved schema contract.

Before the rules are initialised, this section verifies that:

* `active_schema_status` is `APPROVED`;
* `active_schema` is not empty;
* `active_schema_version` is populated;
* every enabled quality rule references columns available in the approved schema.

If the active schema has not been approved, the quality audit must not continue.

### Quality Rule Structure

The quality rules are stored in `quality_rules.json`.

Each rule contains:

* a unique rule ID;
* a category;
* the affected column or columns;
* a severity;
* an enabled status;
* rule-specific parameters;
* a human-readable description.

Supported severity levels are:

```text
WARNING
FAIL
```

A warning identifies a suspicious condition that does not automatically invalidate the row.

A failure identifies a violation that may cause the row to become a quarantine candidate during later processing.

### Initial Quality Contract

The initial rules cover:

* required values;
* blank identifiers;
* duplicate ride IDs;
* permitted categorical values;
* datetime completeness and ordering;
* ride-duration validity;
* missing station information;
* latitude and longitude ranges.

The initial decisions are:

```text
Missing or blank ride_id                    → FAIL
Duplicate ride_id across the dataset        → FAIL
Invalid member_casual value                 → FAIL
Missing started_at or ended_at              → FAIL
ended_at not later than started_at          → FAIL
Ride duration greater than 24 hours         → WARNING
Missing station names or IDs                → WARNING
Coordinates outside global geographic range → FAIL
```

Missing station information is treated as a warning because some ride types may not begin or end at a formal station.

### File-Management Policy

This section follows the same conservative persistence policy used by the schema profiler:

* create `quality_rules.json` when it does not exist;
* preserve an existing valid file;
* add missing top-level sections from defaults;
* add missing default rules without replacing existing customised rules;
* preserve invalid JSON files for human review;
* use in-memory defaults when an existing file cannot be parsed;
* write safe migrations atomically;
* record all creations, migrations, warnings, and failures in `pipeline_audit.log`.

### Expected Outcome

* The approved schema prerequisite is verified.
* `quality_rules.json` exists or safe in-memory defaults are available.
* Existing valid customised rules are preserved.
* Missing required rules are added safely.
* Rule IDs, severities, columns, and parameters are validated.
* Enabled rules reference columns available in the approved schema.
* `quality_rules` is ready for metric generation and assessment in later Part 3 stages.


In [ ]:
# ==========================================
# Part 3.1: Create, Load, and Validate
# Data Quality Rules
# ==========================================

import json
from copy import deepcopy
from datetime import datetime
from pathlib import Path
from typing import Any


# ------------------------------------------
# Validate Part 3.1 dependencies
# ------------------------------------------

REQUIRED_PART_3_1_VARIABLES = {
    "QUALITY_RULES_PATH": QUALITY_RULES_PATH,
    "MASTER_SCHEMA_PATH": MASTER_SCHEMA_PATH,
    "MELBOURNE_TIMEZONE": MELBOURNE_TIMEZONE,
}

for variable_name, variable_value in (
    REQUIRED_PART_3_1_VARIABLES.items()
):
    if variable_value is None:
        raise RuntimeError(
            f"Required Part 3.1 variable is unavailable: "
            f"{variable_name}"
        )


if not isinstance(QUALITY_RULES_PATH, Path):
    raise TypeError(
        "QUALITY_RULES_PATH must be a pathlib.Path object."
    )

if not isinstance(MASTER_SCHEMA_PATH, Path):
    raise TypeError(
        "MASTER_SCHEMA_PATH must be a pathlib.Path object."
    )


# ------------------------------------------
# Reload master registry from disk
# ------------------------------------------

# Reloading ensures that manual schema approval changes made after
# Part 2 are reflected in the current notebook runtime.
try:
    with MASTER_SCHEMA_PATH.open(
        mode="r",
        encoding="utf-8",
    ) as master_schema_file:
        approved_master_registry = json.load(
            master_schema_file
        )

except json.JSONDecodeError as error:
    log_event(
        message=(
            "QUALITY_RULE_INITIALISATION_FAILED: "
            "master_schema.json contains invalid JSON. "
            f"Error={error}"
        ),
        level="CRITICAL",
    )

    raise ValueError(
        "master_schema.json contains invalid JSON."
    ) from error


if not isinstance(approved_master_registry, dict):
    raise TypeError(
        "master_schema.json must contain a JSON object."
    )


# ------------------------------------------
# Validate approved schema prerequisite
# ------------------------------------------

ACTIVE_SCHEMA = approved_master_registry.get(
    "active_schema",
    {},
)

ACTIVE_SCHEMA_VERSION = approved_master_registry.get(
    "active_schema_version"
)

ACTIVE_SCHEMA_STATUS = approved_master_registry.get(
    "active_schema_status"
)


if ACTIVE_SCHEMA_STATUS != "APPROVED":
    log_event(
        message=(
            "QUALITY_RULE_INITIALISATION_BLOCKED: "
            f"active_schema_status={ACTIVE_SCHEMA_STATUS}; "
            "expected=APPROVED."
        ),
        level="CRITICAL",
    )

    raise RuntimeError(
        "Part 3 requires an approved active schema."
    )


if not isinstance(ACTIVE_SCHEMA, dict) or not ACTIVE_SCHEMA:
    log_event(
        message=(
            "QUALITY_RULE_INITIALISATION_BLOCKED: "
            "active_schema is empty or invalid."
        ),
        level="CRITICAL",
    )

    raise RuntimeError(
        "Part 3 requires a non-empty approved active schema."
    )


if (
    not isinstance(ACTIVE_SCHEMA_VERSION, str)
    or not ACTIVE_SCHEMA_VERSION.strip()
):
    log_event(
        message=(
            "QUALITY_RULE_INITIALISATION_BLOCKED: "
            "active_schema_version is missing or invalid."
        ),
        level="CRITICAL",
    )

    raise RuntimeError(
        "Part 3 requires a valid active_schema_version."
    )


# ------------------------------------------
# Define default quality rules
# ------------------------------------------

DEFAULT_QUALITY_RULES = {
    "metadata": {
        "ruleset_version": "1.0.0",
        "ruleset_status": "ACTIVE",
        "source_schema_version": ACTIVE_SCHEMA_VERSION,
        "created_at": None,
        "updated_at": None,
        "description": (
            "Cyclistic source-data quality contract."
        ),
    },
    "rules": {
        "DQ001_RIDE_ID_REQUIRED": {
            "category": "COMPLETENESS",
            "columns": ["ride_id"],
            "severity": "FAIL",
            "enabled": True,
            "rule_type": "NOT_NULL",
            "parameters": {},
            "description": (
                "ride_id must not be null."
            ),
        },
        "DQ002_RIDE_ID_NOT_BLANK": {
            "category": "COMPLETENESS",
            "columns": ["ride_id"],
            "severity": "FAIL",
            "enabled": True,
            "rule_type": "NOT_BLANK",
            "parameters": {},
            "description": (
                "ride_id must not contain an empty or "
                "whitespace-only value."
            ),
        },
        "DQ003_RIDE_ID_UNIQUE": {
            "category": "UNIQUENESS",
            "columns": ["ride_id"],
            "severity": "FAIL",
            "enabled": True,
            "rule_type": "UNIQUE",
            "parameters": {
                "scope": "ALL_FILES",
            },
            "description": (
                "ride_id must be unique across all source files."
            ),
        },
        "DQ004_RIDEABLE_TYPE_ALLOWED": {
            "category": "VALIDITY",
            "columns": ["rideable_type"],
            "severity": "FAIL",
            "enabled": True,
            "rule_type": "ALLOWED_VALUES",
            "parameters": {
                "allowed_values": [
                    "classic_bike",
                    "electric_bike",
                    "docked_bike",
                ],
                "case_sensitive": True,
            },
            "description": (
                "rideable_type must contain an approved "
                "Cyclistic bike category."
            ),
        },
        "DQ005_STARTED_AT_REQUIRED": {
            "category": "COMPLETENESS",
            "columns": ["started_at"],
            "severity": "FAIL",
            "enabled": True,
            "rule_type": "NOT_NULL",
            "parameters": {},
            "description": (
                "started_at must not be null."
            ),
        },
        "DQ006_ENDED_AT_REQUIRED": {
            "category": "COMPLETENESS",
            "columns": ["ended_at"],
            "severity": "FAIL",
            "enabled": True,
            "rule_type": "NOT_NULL",
            "parameters": {},
            "description": (
                "ended_at must not be null."
            ),
        },
        "DQ007_END_AFTER_START": {
            "category": "CONSISTENCY",
            "columns": [
                "started_at",
                "ended_at",
            ],
            "severity": "FAIL",
            "enabled": True,
            "rule_type": "DATETIME_ORDER",
            "parameters": {
                "operator": "GREATER_THAN",
            },
            "description": (
                "ended_at must be later than started_at."
            ),
        },
        "DQ008_DURATION_MAXIMUM": {
            "category": "REASONABLENESS",
            "columns": [
                "started_at",
                "ended_at",
            ],
            "severity": "WARNING",
            "enabled": True,
            "rule_type": "DURATION_MAXIMUM",
            "parameters": {
                "maximum_hours": 24,
            },
            "description": (
                "Ride duration greater than 24 hours requires "
                "review."
            ),
        },
        "DQ009_MEMBER_TYPE_ALLOWED": {
            "category": "VALIDITY",
            "columns": ["member_casual"],
            "severity": "FAIL",
            "enabled": True,
            "rule_type": "ALLOWED_VALUES",
            "parameters": {
                "allowed_values": [
                    "member",
                    "casual",
                ],
                "case_sensitive": True,
            },
            "description": (
                "member_casual must be either member or casual."
            ),
        },
        "DQ010_START_STATION_NAME_MISSING": {
            "category": "COMPLETENESS",
            "columns": ["start_station_name"],
            "severity": "WARNING",
            "enabled": True,
            "rule_type": "NOT_NULL",
            "parameters": {},
            "description": (
                "Missing start station names require review but "
                "may be valid for some ride types."
            ),
        },
        "DQ011_START_STATION_ID_MISSING": {
            "category": "COMPLETENESS",
            "columns": ["start_station_id"],
            "severity": "WARNING",
            "enabled": True,
            "rule_type": "NOT_NULL",
            "parameters": {},
            "description": (
                "Missing start station IDs require review but "
                "may be valid for some ride types."
            ),
        },
        "DQ012_END_STATION_NAME_MISSING": {
            "category": "COMPLETENESS",
            "columns": ["end_station_name"],
            "severity": "WARNING",
            "enabled": True,
            "rule_type": "NOT_NULL",
            "parameters": {},
            "description": (
                "Missing end station names require review but "
                "may be valid for some ride types."
            ),
        },
        "DQ013_END_STATION_ID_MISSING": {
            "category": "COMPLETENESS",
            "columns": ["end_station_id"],
            "severity": "WARNING",
            "enabled": True,
            "rule_type": "NOT_NULL",
            "parameters": {},
            "description": (
                "Missing end station IDs require review but "
                "may be valid for some ride types."
            ),
        },
        "DQ014_START_LAT_RANGE": {
            "category": "VALIDITY",
            "columns": ["start_lat"],
            "severity": "FAIL",
            "enabled": True,
            "rule_type": "NUMERIC_RANGE",
            "parameters": {
                "minimum": -90,
                "maximum": 90,
                "allow_null": False,
            },
            "description": (
                "start_lat must be between -90 and 90."
            ),
        },
        "DQ015_END_LAT_RANGE": {
            "category": "VALIDITY",
            "columns": ["end_lat"],
            "severity": "FAIL",
            "enabled": True,
            "rule_type": "NUMERIC_RANGE",
            "parameters": {
                "minimum": -90,
                "maximum": 90,
                "allow_null": False,
            },
            "description": (
                "end_lat must be between -90 and 90."
            ),
        },
        "DQ016_START_LNG_RANGE": {
            "category": "VALIDITY",
            "columns": ["start_lng"],
            "severity": "FAIL",
            "enabled": True,
            "rule_type": "NUMERIC_RANGE",
            "parameters": {
                "minimum": -180,
                "maximum": 180,
                "allow_null": False,
            },
            "description": (
                "start_lng must be between -180 and 180."
            ),
        },
        "DQ017_END_LNG_RANGE": {
            "category": "VALIDITY",
            "columns": ["end_lng"],
            "severity": "FAIL",
            "enabled": True,
            "rule_type": "NUMERIC_RANGE",
            "parameters": {
                "minimum": -180,
                "maximum": 180,
                "allow_null": False,
            },
            "description": (
                "end_lng must be between -180 and 180."
            ),
        },
    },
    "run_history": [],
}


# ------------------------------------------
# Define required top-level sections
# ------------------------------------------

QUALITY_RULE_SECTION_TYPES = {
    "metadata": dict,
    "rules": dict,
    "run_history": list,
}


REQUIRED_RULE_FIELDS = {
    "category": str,
    "columns": list,
    "severity": str,
    "enabled": bool,
    "rule_type": str,
    "parameters": dict,
    "description": str,
}


SUPPORTED_QUALITY_SEVERITIES = {
    "WARNING",
    "FAIL",
}


# ------------------------------------------
# Set default metadata timestamps
# ------------------------------------------

quality_rules_initialisation_time = datetime.now(
    MELBOURNE_TIMEZONE
).isoformat(timespec="seconds")

DEFAULT_QUALITY_RULES["metadata"][
    "created_at"
] = quality_rules_initialisation_time

DEFAULT_QUALITY_RULES["metadata"][
    "updated_at"
] = quality_rules_initialisation_time


# ------------------------------------------
# Create quality rules file if missing
# ------------------------------------------

def create_quality_rules_if_missing(
    file_path: Path,
) -> bool:
    """
    Create quality_rules.json only when it does not already exist.
    """
    if file_path.exists():

        if not file_path.is_file():
            raise FileExistsError(
                "QUALITY_RULES_PATH exists but is not a file:\n"
                f"{file_path}"
            )

        log_event(
            message=(
                "QUALITY_RULES_FOUND: Existing quality rules "
                "will be preserved."
            ),
            level="INFO",
        )

        return False

    save_json_atomically(
        file_path=file_path,
        data=deepcopy(DEFAULT_QUALITY_RULES),
    )

    if not file_path.exists():
        raise OSError(
            "quality_rules.json was not created:\n"
            f"{file_path}"
        )

    log_event(
        message=(
            "QUALITY_RULES_CREATED: quality_rules.json was "
            "created using the default quality contract."
        ),
        level="INFO",
    )

    return True


# ------------------------------------------
# Load quality rules conservatively
# ------------------------------------------

def load_quality_rules(
    file_path: Path,
) -> tuple[dict, bool]:
    """
    Load quality_rules.json while preserving malformed files.

    Returns:
        rules_data
        loaded_from_disk
    """
    try:
        with file_path.open(
            mode="r",
            encoding="utf-8",
        ) as rules_file:
            loaded_rules = json.load(rules_file)

    except json.JSONDecodeError as error:
        log_event(
            message=(
                "QUALITY_RULES_INVALID_JSON: "
                "The existing file was preserved and in-memory "
                f"defaults will be used. Error={error}"
            ),
            level="CRITICAL",
        )

        return deepcopy(DEFAULT_QUALITY_RULES), False

    except OSError as error:
        log_event(
            message=(
                "QUALITY_RULES_READ_FAILED: "
                "In-memory defaults will be used. "
                f"Error={error}"
            ),
            level="CRITICAL",
        )

        return deepcopy(DEFAULT_QUALITY_RULES), False

    if not isinstance(loaded_rules, dict):
        log_event(
            message=(
                "QUALITY_RULES_INVALID_STRUCTURE: "
                "quality_rules.json must contain a JSON object. "
                "The file was preserved and defaults will be used "
                "in memory."
            ),
            level="CRITICAL",
        )

        return deepcopy(DEFAULT_QUALITY_RULES), False

    return loaded_rules, True


# ------------------------------------------
# Validate top-level quality-rule structure
# ------------------------------------------

def validate_quality_rule_structure(
    rules_data: dict,
) -> tuple[dict, list[str], list[str]]:
    """
    Add missing top-level sections safely.

    Invalid section types are replaced in memory only.
    """
    validated_rules = deepcopy(rules_data)

    missing_sections = []
    invalid_sections = []

    for section_name, expected_type in (
        QUALITY_RULE_SECTION_TYPES.items()
    ):
        if section_name not in validated_rules:
            validated_rules[section_name] = deepcopy(
                DEFAULT_QUALITY_RULES[section_name]
            )

            missing_sections.append(section_name)
            continue

        if not isinstance(
            validated_rules[section_name],
            expected_type,
        ):
            validated_rules[section_name] = deepcopy(
                DEFAULT_QUALITY_RULES[section_name]
            )

            invalid_sections.append(section_name)

    return validated_rules, missing_sections, invalid_sections


# ------------------------------------------
# Add missing default quality rules
# ------------------------------------------

def migrate_missing_quality_rules(
    rules_data: dict,
) -> list[str]:
    """
    Add missing default rules while preserving existing customised
    rules with the same rule IDs.
    """
    missing_rule_ids = []

    configured_rules = rules_data["rules"]

    for rule_id, default_rule in (
        DEFAULT_QUALITY_RULES["rules"].items()
    ):
        if rule_id not in configured_rules:
            configured_rules[rule_id] = deepcopy(
                default_rule
            )

            missing_rule_ids.append(rule_id)

    return missing_rule_ids


# ------------------------------------------
# Validate individual quality rules
# ------------------------------------------

def validate_quality_rule_definitions(
    rules_data: dict,
    approved_schema: dict,
) -> list[dict]:
    """
    Validate quality-rule fields, severity values, and referenced
    approved-schema columns.
    """
    validation_issues = []

    for rule_id, rule_definition in (
        rules_data["rules"].items()
    ):
        if not isinstance(rule_id, str) or not rule_id.strip():
            validation_issues.append(
                {
                    "rule_id": str(rule_id),
                    "issue": "INVALID_RULE_ID",
                }
            )
            continue

        if not isinstance(rule_definition, dict):
            validation_issues.append(
                {
                    "rule_id": rule_id,
                    "issue": "RULE_NOT_OBJECT",
                }
            )
            continue

        for field_name, expected_type in (
            REQUIRED_RULE_FIELDS.items()
        ):
            if field_name not in rule_definition:
                validation_issues.append(
                    {
                        "rule_id": rule_id,
                        "issue": "MISSING_FIELD",
                        "field": field_name,
                    }
                )
                continue

            if not isinstance(
                rule_definition[field_name],
                expected_type,
            ):
                validation_issues.append(
                    {
                        "rule_id": rule_id,
                        "issue": "INVALID_FIELD_TYPE",
                        "field": field_name,
                    }
                )

        severity = rule_definition.get("severity")

        if (
            isinstance(severity, str)
            and severity not in SUPPORTED_QUALITY_SEVERITIES
        ):
            validation_issues.append(
                {
                    "rule_id": rule_id,
                    "issue": "UNSUPPORTED_SEVERITY",
                    "observed": severity,
                }
            )

        enabled = rule_definition.get("enabled")

        if enabled is not True:
            continue

        rule_columns = rule_definition.get(
            "columns",
            [],
        )

        if not isinstance(rule_columns, list):
            continue

        missing_schema_columns = [
            column_name
            for column_name in rule_columns
            if column_name not in approved_schema
        ]

        if missing_schema_columns:
            validation_issues.append(
                {
                    "rule_id": rule_id,
                    "issue": (
                        "COLUMN_NOT_IN_APPROVED_SCHEMA"
                    ),
                    "columns": missing_schema_columns,
                }
            )

    return validation_issues


# ------------------------------------------
# Initialise quality rules
# ------------------------------------------

quality_rules_created = (
    create_quality_rules_if_missing(
        file_path=QUALITY_RULES_PATH,
    )
)

quality_rules_raw, quality_rules_loaded_from_disk = (
    load_quality_rules(
        file_path=QUALITY_RULES_PATH,
    )
)


(
    quality_rules,
    quality_rules_missing_sections,
    quality_rules_invalid_sections,
) = validate_quality_rule_structure(
    rules_data=quality_rules_raw,
)


missing_quality_rule_ids = []

if "rules" not in quality_rules_invalid_sections:
    missing_quality_rule_ids = (
        migrate_missing_quality_rules(
            rules_data=quality_rules,
        )
    )


quality_rule_validation_issues = (
    validate_quality_rule_definitions(
        rules_data=quality_rules,
        approved_schema=ACTIVE_SCHEMA,
    )
)


# ------------------------------------------
# Synchronise ruleset schema version
# ------------------------------------------

stored_source_schema_version = (
    quality_rules["metadata"].get(
        "source_schema_version"
    )
)

if stored_source_schema_version != ACTIVE_SCHEMA_VERSION:
    log_event(
        message=(
            "QUALITY_RULE_SCHEMA_VERSION_UPDATED: "
            f"previous={stored_source_schema_version}; "
            f"current={ACTIVE_SCHEMA_VERSION}."
        ),
        level="WARNING",
    )

    quality_rules["metadata"][
        "source_schema_version"
    ] = ACTIVE_SCHEMA_VERSION

    quality_rules["metadata"][
        "updated_at"
    ] = datetime.now(
        MELBOURNE_TIMEZONE
    ).isoformat(timespec="seconds")


# ------------------------------------------
# Persist safe migrations
# ------------------------------------------

safe_quality_rule_migration_required = (
    quality_rules_loaded_from_disk
    and (
        bool(quality_rules_missing_sections)
        or bool(missing_quality_rule_ids)
        or stored_source_schema_version
        != ACTIVE_SCHEMA_VERSION
    )
    and not quality_rules_invalid_sections
    and not quality_rule_validation_issues
)


if safe_quality_rule_migration_required:
    save_json_atomically(
        file_path=QUALITY_RULES_PATH,
        data=quality_rules,
    )

    log_event(
        message=(
            "QUALITY_RULES_MIGRATION_SAVED: Missing sections, "
            "rules, or schema-version metadata were updated."
        ),
        level="WARNING",
    )


# ------------------------------------------
# Report validation issues
# ------------------------------------------

for issue in quality_rule_validation_issues:
    log_event(
        message=(
            "QUALITY_RULE_VALIDATION_ISSUE: "
            f"{issue}"
        ),
        level="ERROR",
    )


QUALITY_RULES_READY = (
    not quality_rules_invalid_sections
    and not quality_rule_validation_issues
)


if QUALITY_RULES_READY:
    log_event(
        message=(
            "QUALITY_RULES_READY: "
            f"rules={len(quality_rules['rules'])}; "
            f"schema_version={ACTIVE_SCHEMA_VERSION}."
        ),
        level="INFO",
    )


# ------------------------------------------
# Display quality-rule summary
# ------------------------------------------

enabled_quality_rules = {
    rule_id: rule_definition
    for rule_id, rule_definition in (
        quality_rules["rules"].items()
    )
    if rule_definition.get("enabled") is True
}


warning_rule_count = sum(
    1
    for rule in enabled_quality_rules.values()
    if rule.get("severity") == "WARNING"
)

failure_rule_count = sum(
    1
    for rule in enabled_quality_rules.values()
    if rule.get("severity") == "FAIL"
)


print("\n========== Data Quality Rule Initialisation ==========\n")

print(f"Quality rules path       : {QUALITY_RULES_PATH}")
print(f"Quality rules created    : {quality_rules_created}")
print(f"Ruleset version          : {quality_rules['metadata'].get('ruleset_version')}")
print(f"Ruleset status           : {quality_rules['metadata'].get('ruleset_status')}")
print(f"Source schema version    : {quality_rules['metadata'].get('source_schema_version')}")
print(f"Total rules              : {len(quality_rules['rules'])}")
print(f"Enabled rules            : {len(enabled_quality_rules)}")
print(f"Failure rules            : {failure_rule_count}")
print(f"Warning rules            : {warning_rule_count}")
print(f"Validation issues        : {len(quality_rule_validation_issues)}")

print("\nEnabled quality rules:")

for rule_id, rule_definition in (
    enabled_quality_rules.items()
):
    print(
        f"  {rule_id:<30} "
        f"{rule_definition['severity']:<8} "
        f"{rule_definition['rule_type']}"
    )


print("\n" + "=" * 70)

if QUALITY_RULES_READY:
    print("✓ Approved schema prerequisite verified.")
    print("✓ Quality rules created, loaded, or migrated.")
    print("✓ Enabled rules reference approved schema columns.")
    print("✓ Rules are ready for Part 3.2 metric generation.")
else:
    print("✗ Quality rules require correction before Part 3.2.")

print("=" * 70)